In [16]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Optional, Union
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch import Tensor
from datasets import Dataset, DatasetDict, ClassLabel
from sklearn.metrics import accuracy_score, classification_report, f1_score

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    set_seed
)

import tensorrt as trt

In [10]:
set_seed(42)

base_dir: Path = Path("..")
data_dir: Path = base_dir / "data"
models_dir: Path = base_dir / "models"
checkpoints_dir: Path = Path("../output/checkpoints").resolve()
onnx_dir: Path = base_dir / "onnx_export"

for d in (data_dir, models_dir, onnx_dir):
    d.mkdir(parents = True, exist_ok = True)

In [3]:
config: dict[str, Union[int, float, str, bool]] = {
    "model_name": "bert-base-cased", #"microsoft/deberta-v3-base", #"distilbert-base-cased",
    "max_seq_length": 512,
    "bs": 8,
    "gradient_accumulation": 2,
    "epochs": 3,
    "lr": 2e-5,
    "fp16": True,
    "min_tag_frequency": 100, # all tags with less intros are thrown away
}

# Данные

In [4]:
with open(data_dir / "arxivData.json", "r", encoding = "utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} entries")

def ParseRecord(record: dict) -> Optional[dict]:
    try:
        import ast

        title: str = record.get("title", "").strip()
        summary: str = record.get("summary", "").strip()
        tag_str: str = record.get("tag", "[]")
        tag_list: list = ast.literal_eval(tag_str) if tag_str not in ("", "[]", None) else []

        terms = [t["term"] for t in tag_list if isinstance(t, dict) and "term" in t]

        if not title or not terms:
            return None
        
        return {
            "title": title,
            "abstract": summary,
            "tags": terms,
        }
    except (ValueError, SyntaxError, KeyError) as e:
        return None

parsed: list[dict[str, str]] = [r for r in (ParseRecord(rec) for rec in raw_data) if r is not None]
df = pd.DataFrame(parsed)

del parsed, raw_data
print(f"Got {len(df)} valid intros")

Loaded 41000 entries
Got 41000 valid intros


### Предобработка тегов

In [5]:
all_tags = [tag for tags in df["tags"] for tag in tags]
tag_counter = Counter(all_tags)

valid_tags = {tag for tag, cnt in tag_counter.items() if cnt >= config["min_tag_frequency"]}
print(f"Got {len(valid_tags)} after filtering by frequency")

label2id = {tag: idx for idx, tag in enumerate(sorted(valid_tags))}
id2label = {idx: tag for tag, idx in label2id.items()}

Got 50 after filtering by frequency


In [6]:
def GetLabel(tags: list[str]) -> Optional[int]:
    for t in tags:
        if t in label2id:
            return label2id[t]
    return None

df["label"] = df["tags"].apply(GetLabel)
df = df.dropna(subset = {"label"}).copy()
df["label"] = df["label"].astype(int)

mapping_path = data_dir / "label_mapping.json"
with open(mapping_path, "w", encoding = "utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent = 4)

### Токенизация

In [7]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast = False)
print(type(tokenizer))

def TokenizeFN(examples: dict[str, list]) -> dict[str, list]:
    texts = [
        f"{t.strip()} {a.strip()}" if a else t.strip()
        for t, a in zip(examples["title"], examples["abstract"])
    ]
    return tokenizer(texts, truncation = True, max_length = config["max_seq_length"], padding = False)

hf_dataset = Dataset.from_pandas(df[["title", "abstract", "label"]])
hf_dataset = hf_dataset.cast_column("label", ClassLabel(num_classes=len(label2id), names=list(label2id.keys())))
tokenized = hf_dataset.map(
    TokenizeFN,
    batched = True,
    remove_columns = ["title", "abstract"],
    desc = "Tokenizing",
)

train_test = tokenized.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
train_dataset = train_test["train"]

val_test = train_test["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="label")
val_dataset = val_test["train"]
test_dataset = val_test["test"]

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

print(f"train: {len(dataset_dict["train"])}, val: {len(dataset_dict["validation"])}, test: {len(dataset_dict["test"])}")

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>


Casting the dataset:   0%|          | 0/41000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/41000 [00:00<?, ? examples/s]

train: 32800, val: 4100, test: 4100


# Модель

In [12]:
def ComputeMetrics(eval_pred: EvalPrediction) -> dict[str, float]:
    logits: np.ndarray = eval_pred.predictions
    labels: np.ndarray = eval_pred.label_ids
    predictions: np.ndarray = np.argmax(logits, axis = -1)
    accuracy: float = float(np.mean(predictions == labels))
    return {"accuracy": accuracy}

data_collator: DataCollatorWithPadding = DataCollatorWithPadding(
    tokenizer = tokenizer,
    padding = True,
    # max_length = config["max_seq_length"]
)

total_train_steps: int = (len(dataset_dict["train"]) // config["bs"]) * config["epochs"]
warmup_steps: int = int(total_train_steps * 0.1)

model: AutoModelForSequenceClassification = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path = config["model_name"],
    num_labels = len(label2id),
    id2label = id2label,
    label2id = label2id,
    problem_type = "single_label_classification",
    ignore_mismatched_sizes = True,
)

training_args: TrainingArguments = TrainingArguments(
    output_dir = str(checkpoints_dir),
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = config["lr"],
    per_device_train_batch_size = config["bs"],
    per_device_eval_batch_size = config["bs"] * 2,
    num_train_epochs = config["epochs"],
    weight_decay = 0.05,
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy",
    bf16 = True,
    fp16 = False,
    save_total_limit = 1,
    warmup_steps = warmup_steps,
    label_smoothing_factor = 0.1,
    lr_scheduler_type = "cosine",
    dataloader_num_workers = 4,
    dataloader_prefetch_factor = 2,
)

trainer: Trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = dataset_dict["train"],
    eval_dataset = dataset_dict["validation"],
    data_collator = data_collator,
    # tokenizer = tokenizer,
    compute_metrics = ComputeMetrics,
)

train_metrics: dict[str, float] = trainer.train()
trainer.save_model()
tokenizer.save_pretrained(save_directory = str(checkpoints_dir))

print(f"Training complete. Final metrics: {train_metrics}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,1.674600,1.659794,0.652927
2,1.506100,1.587431,0.678293
3,1.355900,1.599413,0.684146


Training complete. Final metrics: TrainOutput(global_step=12300, training_loss=1.6205972066739711, metrics={'train_runtime': 802.3985, 'train_samples_per_second': 122.632, 'train_steps_per_second': 15.329, 'total_flos': 1.717284531196608e+16, 'train_loss': 1.6205972066739711, 'epoch': 3.0})


# Экспорт в ONNX

In [15]:
!optimum-cli export onnx \
    --model ../output/checkpoints \
    --task text-classification \
    ../output/onnx

`torch_dtype` is deprecated! Use `dtype` instead!
/home/user/dev/YSDA-1/ml_env/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


In [ ]:
# onnx_dir: Path = base_dir / "output/onnx"
# onnx_dir.mkdir(parents = True, exist_ok = True)
# onnx_model_path: Path = onnx_dir / "model.onnx"

# best_model_path: str = str(checkpoints_dir)
# print(f"Loading model from {best_model_path}")

# export_model: AutoModelForSequenceClassification = AutoModelForSequenceClassification.from_pretrained(
#     pretrained_model_name_or_path = best_model_path,
#     num_labels = len(label2id),
#     id2label = id2label,
#     label2id = label2id,
#     problem_type = "single_label_classification",
# )
# export_tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(best_model_path)

# export_model.eval()
# export_model.to(device = "cpu")

# dummy_text: str = "Dummy text for getting the shape"
# dummy_input: dict[str, Tensor] = export_tokenizer(
#     dummy_text,
#     return_tensors = "pt",
#     truncation = True,
#     max_length = config["max_seq_length"],
# )
# input_ids: Tensor = dummy_input["input_ids"]
# attention_mask: Tensor = dummy_input["attention_mask"]

# # with torch.no_grad():
# #     test_output = export_model(input_ids = input_ids, attention_mask = attention_mask),
# #     print(f"Model output keys: {test_output.keys()}")
# #     print(f"Logits shape: {test_output.logits.shape}")

# print(f"Exporting model to ONNX: {onnx_model_path}")

# torch.onnx.export(
#     model = export_model,
#     args = (input_ids, attention_mask),
#     f = str(onnx_model_path),
#     # opset_version = 18,
#     export_params = True,
#     input_names = ["input_ids", "attention_mask"],
#     output_names = ["logits"],
#     dynamic_axes = {
#         "input_ids": {0: "batch_size", 1: "sequence_length"},
#         "attention_mask": {0: "batch_size", 1: "sequence_length"},
#         "logits": {0: "batch_size"},
#     },
#     do_constant_folding = True,
#     verbose = False,
# )

# print(f"ONNX model exported to {onnx_model_path}")
# print(f"  Size: {onnx_model_path.stat().st_size / 1024 / 1024:.2f} MB")


Loading model from /home/user/dev/YSDA-1/ml-2/hw4/output/checkpoints
Exporting model to ONNX: ../output/onnx/model.onnx


/tmp/ipykernel_1379208/2556836434.py:37: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0409 12:04:26.598000 1379208 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0409 12:04:26.599000 1379208 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0409 12:04:26.600000 1379208 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale:

Applied 71 of general pattern rewrite rules.
ONNX model exported to ../output/onnx/model.onnx
  Size: 1.29 MB


# Экспорт в Tensor-RT

In [ ]:
onnx_model_path: Path = Path("../output/onnx/model.onnx").resolve()
engine_path: Path = Path("../models/model.engine").resolve()
Path("../models").mkdir(parents = True, exist_ok = True)

trt_logger: trt.Logger = trt.Logger(trt.Logger.WARNING)
builder: trt.Builder = trt.Builder(trt_logger)
network: trt.INetworkDefinition = builder.create_network(
    1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
)
parser: trt.IOnnxParser = trt.OnnxParser(network, trt_logger)

with open(onnx_model_path, "rb") as f:
    if not parser.parse(f.read()):
        for error_index in range(parser.num_errors):
            print(parser.get_error(error_index))
        raise RuntimeError("ONNX parsing failed")

config: trt.IBuilderConfig = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
config.set_flag(trt.BuilderFlag.FP16)

profile: trt.IOptimizationProfile = builder.create_optimization_profile()
profile.set_shape("input_ids", (1, 1), (1, 64), (8, 256))
profile.set_shape("attention_mask", (1, 1), (1, 64), (8, 256))
profile.set_shape("token_type_ids", (1, 1), (1, 64), (8, 256))
config.add_optimization_profile(profile)

engine: bytes = builder.build_serialized_network(network, config)
if engine is None:
    raise RuntimeError("TensorRT engine build failed.")

with open(engine_path, "wb") as f:
    f.write(engine)

print(f"Engine saved to {engine_path}")
print(f"Size: {Path(engine_path).stat().st_size / 1024 / 1024:.2f} MB")

[04/09/2026-12:49:36] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[04/09/2026-12:49:37] [TRT] [W] ModelImporter.cpp:739: Make sure input input_ids has Int64 binding.
[04/09/2026-12:49:37] [TRT] [W] ModelImporter.cpp:739: Make sure input attention_mask has Int64 binding.
[04/09/2026-12:49:37] [TRT] [W] ModelImporter.cpp:739: Make sure input token_type_ids has Int64 binding.
✓ Engine saved to /home/user/dev/YSDA-1/ml-2/hw4/models/model.engine
  Size: 414.96 MB
